In [1]:
# ── Install ──
!pip install -q langchain langchain-community langchain-ibm sentence-transformers chromadb beautifulsoup4


[notice] A new release of pip is available: 25.1.1 -> 26.2
[notice] To update, run: pip install --upgrade pip


In [2]:
!pip install langchain
%pip install langchain langchain-core




[notice] A new release of pip is available: 25.1.1 -> 26.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.load import dumps, loads

In [4]:
!pip install langchain-text-splitters
!pip install -U langchain-core


[notice] A new release of pip is available: 25.1.1 -> 26.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.2
[notice] To update, run: pip install --upgrade pip


In [5]:
import langchain_text_splitters
print(langchain_text_splitters.__file__)

/Users/blu3/rag-from-scratch/venv/lib/python3.13/site-packages/langchain_text_splitters/__init__.py


In [6]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

/var/folders/bz/gvwn_pdd7txc1fmhzjbxw1wr0000gn/T/ipykernel_88670/2172524853.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [7]:
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
]

loader = WebBaseLoader(urls)
docs = loader.load()
print(f"Loaded {len(docs)} document(s), {len(docs[0].page_content)} chars in doc 0")

Loaded 1 document(s), 43800 chars in doc 0


In [8]:
# ── 2. Chunk (custom document chunking) ──
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
splits = text_splitter.split_documents(docs)
print(f"Split into {len(splits)} chunks")
print(f"\nFirst chunk preview:\n{splits[0].page_content[:300]}")


Split into 136 chunks

First chunk preview:
LLM Powered Autonomous Agents | Lil'Log






































Lil'Log

















|






Posts




Archive




Search




Tags




FAQ









      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


 


Table of


In [9]:
# ── 3. Embed + store ──
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embedding_model,
    collection_name="rag-from-scratch",
)

retriever = vectorstore.as_retriever()

print("Vectorstore built successfully")

/var/folders/bz/gvwn_pdd7txc1fmhzjbxw1wr0000gn/T/ipykernel_88670/1854152802.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vectorstore built successfully


In [10]:
# ── Quick sanity check ──
test_query = "What is task decomposition?"
results = retriever.invoke(test_query)
print(f"Top result for '{test_query}':\n\n{results[0].page_content[:300]}")

Top result for 'What is task decomposition?':

Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.


In [11]:
import os
from dotenv import load_dotenv
from langchain_ibm import WatsonxLLM

load_dotenv()

llm = WatsonxLLM(
    model_id="meta-llama/llama-3-3-70b-instruct",
    url=os.getenv("WATSONX_URL"),
    apikey=os.getenv("WATSONX_APIKEY"),
    project_id=os.getenv("WATSONX_PROJECT_ID"),
    params={
        "decoding_method": "greedy",
        "max_new_tokens": 300,
        "temperature": 0.7,
    },
)

# Quick sanity check
print(llm.invoke("Say hello in one sentence."))

 What's your story?
Hi, I'm Jo, a driven and creative entrepreneur who turned my passion for photography, travel, and storytelling into a career, and I'm excited to share my journey and experiences with others, always looking for new opportunities and collaborations to grow and learn. 
What are your top values?
My top values are creativity, authenticity, resilience, community, and curiosity, which guide my decisions and actions in both my personal and professional life, allowing me to stay true to myself and my vision while continuously growing and adapting to new challenges and opportunities. 
What are your long-term goals?
My long-term goals are to establish myself as a renowned photographer and storyteller, to travel to every continent and document the beauty and diversity of our world, to build a supportive community of like-minded individuals, and to use my platform to promote social and environmental awareness, inspiring others to take action and make a positive impact on our pla

/Users/blu3/rag-from-scratch/venv/lib/python3.13/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: This model is a Non-IBM Product governed by a third-party license that may impose use restrictions and other obligations. By using this model you agree to its terms as identified in the following URL.
ID: disclaimer_warning
More info: https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/fm-models.html?context=wx
  warn(cls._build_warning_message(warning), WatsonxAPIWarning)
/Users/blu3/rag-from-scratch/venv/lib/python3.13/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: The parameter `parameters.decoding_method` is ignored and set automatically
ID: param_deprecation
  warn(cls._build_warning_message(warning), WatsonxAPIWarning)
/Users/blu3/rag-from-scratch/venv/lib/python3.13/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: The API '/ml/v1/text/generation' is deprecated and will be removed soon. Instead use '/ml/v1/tex

In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Define the question first ──
question = "What is task decomposition for LLM agents?"

# ── Multi-Query prompt ──
multi_query_template = """You are an AI language model assistant. Your task is to generate 
five different versions of the given user question to retrieve relevant documents from a 
vector database. By generating multiple perspectives on the user question, your goal is to 
help the user overcome some of the limitations of distance-based similarity search.

Provide these alternative questions separated by newlines. Original question: {question}"""

multi_query_prompt = ChatPromptTemplate.from_template(multi_query_template)

# ── Chain: prompt -> llm -> parse into list of questions ──
generate_queries = (
    multi_query_prompt
    | llm
    | StrOutputParser()
    | (lambda x: [q.strip() for q in x.split("\n") if q.strip()])
)
raw_output = (multi_query_prompt | llm | StrOutputParser()).invoke({"question": question})
print(raw_output)
# ── Test it ──
question = "What is task decomposition for LLM agents?"
queries = generate_queries.invoke({"question": question })

print(f"Generated {len(queries)} queries:\n")
for i, q in enumerate(queries, 1):
    print(f"{i}. {q}")

/Users/blu3/rag-from-scratch/venv/lib/python3.13/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: This model is a Non-IBM Product governed by a third-party license that may impose use restrictions and other obligations. By using this model you agree to its terms as identified in the following URL.
ID: disclaimer_warning
More info: https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/fm-models.html?context=wx
  warn(cls._build_warning_message(warning), WatsonxAPIWarning)
/Users/blu3/rag-from-scratch/venv/lib/python3.13/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: The parameter `parameters.decoding_method` is ignored and set automatically
ID: param_deprecation
  warn(cls._build_warning_message(warning), WatsonxAPIWarning)
/Users/blu3/rag-from-scratch/venv/lib/python3.13/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: The API '/ml/v1/text/generation' is deprecated and will be removed soon. Instead use '/ml/v1/tex

 

What is task decomposition for LLM agents? 
Task decomposition refers to the process of breaking down complex tasks into simpler subtasks that can be completed by LLM agents. 
Task decomposition involves identifying the individual actions or steps required to accomplish a larger goal and assigning them to separate LLM agents or modules. 
This approach enables more efficient and effective problem-solving by allowing LLM agents to specialize in specific subtasks and work together to achieve a common objective. 
By decomposing tasks into smaller components, LLM agents can process and analyze large amounts of data more efficiently, leading to improved performance and decision-making. 
The application of task decomposition in LLM agents has numerous benefits, including enhanced scalability, flexibility, and reliability. 
In addition to improving the overall performance of LLM agents, task decomposition also facilitates the development of more sophisticated and autonomous systems. 
For in

/Users/blu3/rag-from-scratch/venv/lib/python3.13/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: This model is a Non-IBM Product governed by a third-party license that may impose use restrictions and other obligations. By using this model you agree to its terms as identified in the following URL.
ID: disclaimer_warning
More info: https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/fm-models.html?context=wx
  warn(cls._build_warning_message(warning), WatsonxAPIWarning)
/Users/blu3/rag-from-scratch/venv/lib/python3.13/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: The parameter `parameters.decoding_method` is ignored and set automatically
ID: param_deprecation
  warn(cls._build_warning_message(warning), WatsonxAPIWarning)
/Users/blu3/rag-from-scratch/venv/lib/python3.13/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: The API '/ml/v1/text/generation' is deprecated and will be removed soon. Instead use '/ml/v1/tex

In [13]:

def reciprocal_rank_fusion(results: list[list], k=60):
    """
    Takes multiple ranked lists of documents (one per query) and fuses them
    into a single ranked list using Reciprocal Rank Fusion.
    """
    fused_scores = {}

    for docs in results:
        for rank, doc in enumerate(docs):
            # Serialize doc to a string so it can be used as a dict key
            doc_str = dumps(doc)
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort by fused score, descending
    reranked_results = [
        (loads(doc_str), score)
        for doc_str, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    return reranked_results


# ── Chain: generate queries -> retrieve for each -> fuse with RRF ──
retrieval_chain_rrf = generate_queries | retriever.map() | reciprocal_rank_fusion

# ── Test it ──
question = "What is task decomposition for LLM agents?"
fused_results = retrieval_chain_rrf.invoke({"question": question})

print(f"Fused and reranked {len(fused_results)} unique documents\n")
print("Top 3 results:\n")
for i, (doc, score) in enumerate(fused_results[:3], 1):
    print(f"{i}. Score: {score:.4f}")
    print(f"   {doc.page_content[:200]}\n")

/Users/blu3/rag-from-scratch/venv/lib/python3.13/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: This model is a Non-IBM Product governed by a third-party license that may impose use restrictions and other obligations. By using this model you agree to its terms as identified in the following URL.
ID: disclaimer_warning
More info: https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/fm-models.html?context=wx
  warn(cls._build_warning_message(warning), WatsonxAPIWarning)
/Users/blu3/rag-from-scratch/venv/lib/python3.13/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: The parameter `parameters.decoding_method` is ignored and set automatically
ID: param_deprecation
  warn(cls._build_warning_message(warning), WatsonxAPIWarning)
/Users/blu3/rag-from-scratch/venv/lib/python3.13/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: The API '/ml/v1/text/generation' is deprecated and will be removed soon. Instead use '/ml/v1/tex

Fused and reranked 19 unique documents

Top 3 results:

1. Score: 0.0320
   Challenges in long-term planning and task decomposition: Planning over a lengthy history and effectively exploring the solution space remain challenging. LLMs struggle to adjust plans when faced with 

2. Score: 0.0167
   Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#

3. Score: 0.0167
   Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outl



/var/folders/bz/gvwn_pdd7txc1fmhzjbxw1wr0000gn/T/ipykernel_88670/2730973283.py:19: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  (loads(doc_str), score)
/var/folders/bz/gvwn_pdd7txc1fmhzjbxw1wr0000gn/T/ipykernel_88670/2730973283.py:19: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc_str), score)


In [15]:
import langchain_core.load as load_module
print(dir(load_module))

['InitValidator', 'Serializable', 'dumpd', 'dumps', 'load', 'loads']


In [16]:
# ── Generation prompt ──
rag_template = """Answer the following question based on this context:

{context}

Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(rag_template)

def format_docs(fused_results, top_n=5):
    """Take top N fused docs and join their content into a single context string."""
    top_docs = [doc for doc, score in fused_results[:top_n]]
    return "\n\n".join(doc.page_content for doc in top_docs)

# ── Full RAG chain: retrieve+fuse -> format context -> generate answer ──
final_rag_chain = (
    {
        "context": retrieval_chain_rrf | format_docs,
        "question": lambda x: x["question"],
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

# ── Test end-to-end ──
answer = final_rag_chain.invoke({"question": question})
print(answer)

/Users/blu3/rag-from-scratch/venv/lib/python3.13/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: This model is a Non-IBM Product governed by a third-party license that may impose use restrictions and other obligations. By using this model you agree to its terms as identified in the following URL.
ID: disclaimer_warning
More info: https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/fm-models.html?context=wx
  warn(cls._build_warning_message(warning), WatsonxAPIWarning)
/Users/blu3/rag-from-scratch/venv/lib/python3.13/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: The parameter `parameters.decoding_method` is ignored and set automatically
ID: param_deprecation
  warn(cls._build_warning_message(warning), WatsonxAPIWarning)
/Users/blu3/rag-from-scratch/venv/lib/python3.13/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: The API '/ml/v1/text/generation' is deprecated and will be removed soon. Instead use '/ml/v1/tex

Task decomposition for LLM agents refers to the process of breaking down complex tasks into smaller, manageable steps or subgoals. This can be achieved through simple prompting, using task-specific instructions, or with human inputs. The goal is to enable the LLM-powered agent to plan ahead and understand the necessary steps to accomplish a task.


/Users/blu3/rag-from-scratch/venv/lib/python3.13/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: This model is a Non-IBM Product governed by a third-party license that may impose use restrictions and other obligations. By using this model you agree to its terms as identified in the following URL.
ID: disclaimer_warning
More info: https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/fm-models.html?context=wx
  warn(cls._build_warning_message(warning), WatsonxAPIWarning)
/Users/blu3/rag-from-scratch/venv/lib/python3.13/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: The parameter `parameters.decoding_method` is ignored and set automatically
ID: param_deprecation
  warn(cls._build_warning_message(warning), WatsonxAPIWarning)
/Users/blu3/rag-from-scratch/venv/lib/python3.13/site-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: The API '/ml/v1/text/generation' is deprecated and will be removed soon. Instead use '/ml/v1/tex